In [1]:
from collections import defaultdict
from matplotlib.colors import LogNorm
from pathlib import Path
import jenkspy
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
import shutil
import string

from _configs.country_config import *
from _configs.files_config import *
from _configs.run_config import *

from _utils.utils import *

Go to root directory

In [2]:
if Path.cwd().name == "population_validation":
    os.chdir(Path.cwd().parent)

print(f"Current Directory: {Path.cwd()}")

Current Directory: /work/tuv89272/Calibration_Pipeline_v7_NG-SE


# Validate model growth rate

In [3]:
birth_rate_str = f"{birth_rate:.4f}"

run_folder = Path(pre_calibration_run_path)
run_folder.mkdir(exist_ok=True)

run_inputs_folder = Path(PRE_CALIBRATION_RUN_INPUTS_DIR)

# delete any yaml files in the run input folder to avoid confusion 
for file in run_inputs_folder.glob("*.yaml"):
    file.unlink()

# copy bin folder from template to pre_calibration path
template_bin = os.path.join(template_path, "bin")
run_bin = os.path.join(run_folder, "bin")

if os.path.exists(template_bin):
    if os.path.exists(run_bin):
        info(f"Bin folder already exists in run path: {run_bin}. Skipping copy.")
    else:
        shutil.copytree(template_bin, run_bin, copy_function=shutil.copy2, symlinks=True)
        ok(f"Copied bin folder from {template_bin} to {run_bin}")

→ Bin folder already exists in run path: _pre_calibration_population_growth_rate_validation_3/bin. Skipping copy.


Copy input config yml to pre calibration run folder

<span style="color:red">Make sure to use correct district/district_seq1 file version!</span>

In [4]:
TEMPLATE = "input_population_bins.yml"

run_input_dir = Path(run_folder) / "input"
run_input_dir.mkdir(exist_ok=True)

output_dir = Path(run_folder) / "output"
output_dir.mkdir(exist_ok=True)

run_folder_full_path = os.path.abspath(run_folder)
print(f"Run folder full path: {run_folder_full_path}")

with open(TEMPLATE, "r", encoding="utf-8") as f:
    input_template_text = f.read()    

out_text = input_template_text.replace(f"_initialpopulation#POPULATION#.asc", initial_population_projected_raster_path.removeprefix(f"{generated_data_path}/{country_code}"))
out_text = out_text.replace("#BETA#", "_zero")
out_text = out_text.replace("#ACCESS_RATE#", "")
out_text = out_text.replace("#BIRTH_RATE#", birth_rate_str)
out_text = out_text.replace("#CALIBRATION_YEAR#", f"{calibration_year}")
out_text = out_text.replace("#COUNTRY_CODE#", f"{country_code}")
out_text = out_text.replace("#INITIAL_YEAR#", f"{initial_year}")
out_text = out_text.replace("#INPUT_PATH#", "input")
out_text = out_text.replace("#POPULATION_SCALE#", f"{pre_calibration_population_validation_scale}")
out_text = out_text.replace("initialpopulation#POPULATION#", f"initpopulation_{initial_year}_inferred_for_sim")
out_text = out_text.replace(f"{country_code}_seasonality_1_location", f"{country_code}_seasonality")
out_text = out_text.replace(f"{country_code}_treatmentseeking", f"{country_code}_treatment_seeking_normalized")
out_name = os.path.abspath(os.path.join(run_input_dir, f"input_population_bins_beta_zero_pop_{pre_calibration_population_validation_scale}.yml"))

# out_text = out_text.replace("district.asc", "districts.asc")
out_text = out_text.replace("district.asc", "district_seq1.asc")

with open(out_name, "w", encoding="utf-8") as outf:
    outf.write(out_text)
    print(f"Created input file: {out_name}\n")
    
# Copy other raster files that are needed for the model but not generated here (e.g. population, districts, traveltime, etc.)

# print wd
print(f"Current working directory: {os.getcwd()}")
other_files = [
    zero_beta_raster_path,
    initial_population_projected_raster_path,
    districts_raster_sequential_path,
    travel_time_raster_path,
    treatment_seeking_raster_path,
    ]
for o_file in other_files:
    if os.path.exists(o_file):
        # print(f"\nCopying {o_file} to {VALIDATION_RUN_INPUTS_DIR}")
        dest = os.path.abspath(os.path.join(run_input_dir, o_file.replace(f"{data_path}/", "").replace(f"{calibration_analysis_path}/", "").replace(f"{generated_data_path}/", "")))
        with open(o_file, "r", encoding="utf-8") as srcf:
            with open(dest, "w", encoding="utf-8") as destf:

                destf.write(srcf.read())
        ok(f"Copied {o_file} to {dest}")
    else:
        warn(f"Warning: required raster not found: {o_file}. Please ensure it is available in the current directory.") 

Run folder full path: /work/tuv89272/Calibration_Pipeline_v7_NG-SE/_pre_calibration_population_growth_rate_validation_3
Created input file: /work/tuv89272/Calibration_Pipeline_v7_NG-SE/_pre_calibration_population_growth_rate_validation_3/input/input_population_bins_beta_zero_pop_0.25.yml

Current working directory: /work/tuv89272/Calibration_Pipeline_v7_NG-SE
✓ Copied generated/ng-se_beta_zero.asc to /work/tuv89272/Calibration_Pipeline_v7_NG-SE/_pre_calibration_population_growth_rate_validation_3/input/ng-se_beta_zero.asc
✓ Copied generated/ng-se_population_backwards_projected_2011_20.64M.asc to /work/tuv89272/Calibration_Pipeline_v7_NG-SE/_pre_calibration_population_growth_rate_validation_3/input/ng-se_population_backwards_projected_2011_20.64M.asc
✓ Copied DATA/ng-se_district_seq1.asc to /work/tuv89272/Calibration_Pipeline_v7_NG-SE/_pre_calibration_population_growth_rate_validation_3/input/ng-se_district_seq1.asc
✓ Copied DATA/ng-se_traveltime.asc to /work/tuv89272/Calibration_Pipeli

### Create run.sh script

In [5]:
# clean 0.log, malaSim_0.pid, and run.sh files if they exist
for file_name in ["0.log", "malaSim_0.pid", "run.sh"]:
    file_path = run_folder / file_name
    if file_path.exists():
        file_path.unlink()
        print(f"Removed existing file: {file_path}")

Removed existing file: _pre_calibration_population_growth_rate_validation_3/run.sh


In [6]:
run_sh_code = f"""#!/bin/bash

cd {run_folder_full_path} || exit 1

nohup ./bin/MalaSim \\
-i input/input_population_bins_beta_zero_pop_{pre_calibration_population_validation_scale}.yml \\
-r SQLiteMonthlyReporter \\
-o output/pop_validation \\
-j 0 \\
-v 1 \\
> 0.log 2>&1 &

echo $! > malaSim_0.pid
echo "MalaSim started in background"
echo "PID: $(cat malaSim_0.pid)"
echo "Log: 0.log"
"""

run_sh_path = run_folder / "run.sh"
run_sh_path.write_text(run_sh_code)
run_sh_path.chmod(0o755)  # make it executable

info(f"Written to: {run_sh_path} with scale {pre_calibration_population_validation_scale}")

→ Written to: _pre_calibration_population_growth_rate_validation_3/run.sh with scale 0.25


Run <code>./run.sh</code> in terminal